# 🔐 **Security Evaluation of a Face Recognition System**

## Pre-Processing Defense Strategy Implementation for NN1

**Academic Year:** 2024-2025  
**Group:** 04

---

### 👥 Team Members
- **Agostino Cardamone** — `0622702276`
- **Asja Antonucci**     — `0622702437`
- **Chiara Ferraioli**   — `0622702169`

---

### 📚 Overview of This Section

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Pre-Processing Defense Strategy](#2-pre-processing-defense-strategy)

## 1. Setup and Data Loading

#### Environment Setup

To ensure reproducibility and avoid package conflicts, it is strongly recommended to run all experiments in an isolated environment. We use Conda to create and manage the project environment, and all Python dependencies are listed in the requirements.txt file.

In [ ]:
# 1) Create a new environment named “aic_env”
#conda create -n aic_env python=3.10 -y

# 2) Switch into the new environment
# On Windows:
# conda activate aic_env
# On Linux/macOS:
# source activate aic_env

# 3) Install all dependencies
#!pip install -r requirements.txt

import os 

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("torch.version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

#### Dataset Configuration and Paths

This section defines the core paths and settings used throughout the project to manage the dataset structure and preprocessing behavior.

- `dataset_dir` points to the root directory containing the dataset files.
- `dataset_selection` controls whether to regenerate the test set from scratch (⚠️set to True only if you have extracted `vggface2_train` inside `dataset/vggface2_train/trainset`⚠️)
- `mtcnn_processing_nn1` determines whether to apply MTCNN face alignment for NN1 preprocessing.
- `test_set_rnd` specifies whether the test set should be built randomly or from the pre-defined class list in `test_set.csv`.

Metadata and folder structure:
- `vgg2_dataset_annotations_path` points to `identity_meta.csv`, which contains class metadata (name, gender, etc.).
- `test_set_data_folder` is the location of test samples.
- `test_set_annotations_folder` contains the CSV file describing the test set structure.

All experiment outputs will be saved to:
- `results_folder` — for accuracy results and evaluations.
- `adversarial_folder` — for storing generated adversarial images.

The variable `device` automatically selects GPU if available, otherwise defaults to CPU.

In [ ]:
from utils import *         # Project-specific utilities and imports

# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base directory containing all dataset-related files
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# If True, a new test set will be built by sampling and copying images from the original VGGFace2 dataset
# NOTE: This requires the dataset to be downloaded and extracted under 'vggface2_train/trainset'
# If False, the existing CSV files will be loaded without modifying or copying any images
dataset_selection = False  

# If True, test images will be aligned and cropped using MTCNN preprocessing (for NN1 compatibility)
mtcnn_processing_nn1 = False

# If True, the test set will be built via random sampling of identities and images
# If False, the test set will be built based on predefined class IDs listed in 'test_set.csv'
test_set_rnd = False

# Path to VGGFace2 identity metadata (includes Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set files and images
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')

# Directory to store evaluation results (e.g., SEC curves)
results_folder = os.path.join(os.getcwd(), 'results')

# Directory to save generated adversarial examples
adversarial_folder = os.path.join(os.getcwd(), 'attacks')

# Device configuration: use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


#### Load Test Set and Class Labels

This block performs two critical initializations:

1. **Load test set metadata**
   - The file `test_set.csv` is read into a DataFrame.
   - It contains exactly 100 test identities, each with 10 face images located in `testset/samples/`.
   - The total number of expected images is computed as `100 × 10 = 1000`.

2. **Load class label mappings**
   - The face recognition model requires access to the full list of class names (8631 identities).
   - These are loaded from a `.npy` file originally published by the official [`rcmalli/keras-vggface`](https://github.com/rcmalli/keras-vggface) repository.
   - If the file is not present locally, it is automatically downloaded.
   - Any surrounding whitespace in label strings is stripped to ensure clean formatting.

This step is necessary to convert model outputs (class indices) into readable identity labels.

In [ ]:
# —————————————————————————————————————————————
#              Load annotation CSVs
# —————————————————————————————————————————————

# Read the existing test_set.csv describing our 100 test identities
# (each identity will have 10 samples in the folder structure)
test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

# Calculate how many total images we expect in the test set:
# number of identities × 10 images each
test_set_size = len(test_set) * 10

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

#### Load and Prepare Face Recognition Model (NN1)

This section sets up the face recognition model referred to as **NN1**, based on the `InceptionResnetV1` architecture provided by the `facenet-pytorch` library.

- The model is initialised with weights pre-trained on the **VGGFace2** dataset, which contains over 8,000 people identities.
- It is set to evaluation mode (`.eval()`), disabling stochastic layers such as dropout and batch normalisation updates, ensuring deterministic inference.
- By default, the model outputs a 512-dimensional feature embedding for each face. However, enabling the `.classify = True` flag appends a classification head, allowing the model to directly output class logits over the **8,631 identities** present in the VGGFace2 training set.

In [ ]:
from facenet_pytorch import InceptionResnetV1

# ——————————————————————————————————————————————————————
#   Initialize the pre-trained face-recognition model
# ——————————————————————————————————————————————————————

# We use the InceptionResnetV1 architecture from the facenet-pytorch package,
# pre-trained on the VGGFace2 dataset for high-quality face embeddings.
# By calling .eval(), we set the model to inference mode (disables dropout, batchnorm updates).
# We then move the model to the appropriate device (GPU if available, else CPU).
nn1 = InceptionResnetV1(
    pretrained='vggface2'  # load weights trained on the VGGFace2 face dataset
).eval().to(device)         # switch to evaluation mode and transfer to GPU/CPU

# —————————————————————————————————————————————
#           Enable classification head
# —————————————————————————————————————————————

# By default, InceptionResnetV1 returns 512-dimensional embeddings.
# Setting .classify instructs the model to append a linear classification
# layer on top of the embeddings, so that nn1(input) returns raw class logits
# for all identities in VGGFace2 (8 631 classes), instead of embeddings.
nn1.classify = True

#### Face Detection and Alignment (MTCNN)

To ensure consistent and high-quality input for face recognition, the **MTCNN** (Multi-task Cascaded Convolutional Networks) model is used as a preprocessing stage. MTCNN carries out multiple tasks in sequence to detect and align faces with a high degree of accuracy. Specifically, it:

- Locates the most prominent face within each image.
- Aligns facial features based on detected landmarks (e.g., eyes, nose, and mouth).
- Crops and resizes the face region to a fixed resolution of **160×160 pixels**, matching the expected input size of the recognition model.

The aligned faces are returned as PyTorch tensors, ready for immediate use in model inference. Additionally, these preprocessed images can be optionally saved to disk, allowing the system to bypass real-time face detection in subsequent runs — a significant benefit in terms of efficiency.

> When the flag `mtcnn_processing_nn1` is set to `True`, the aligned face crops are automatically stored in the directory:  
> `testset/cropped_faces_nn1/{class_name}/img_XXX.jpg`

In [ ]:
# ———————————————————————————————————————————————————
#  Initialize the face detector and aligner (MTCNN)
# ———————————————————————————————————————————————————
# We use MTCNN from facenet-pytorch to detect, crop, and align faces in one step.
# When you call face_detector_nn1(img_batch), it returns a tensor of shape [B, 3, image_size, image_size]
# containing the aligned face crops, ready to feed into nn1 or an adversarial attack.

face_detector_nn1 = MTCNN(
    image_size=160,                             # int: output height/width of each face crop (default=160)
    margin=0,                                   # int: number of pixels to expand the face bounding box (default=0)
    min_face_size=20,                           # int: minimum face size (in pixels) that the detector will attempt to locate (default=20)
    thresholds=[0.6, 0.7, 0.7],                 # list of 3 floats: score thresholds for each detection stage—
                                                #   P-Net, R-Net, and O-Net respectively (default=[0.6, 0.7, 0.7])
    factor=0.709,                               # float: scale factor between pyramid levels; controls the search granularity (default=0.709)
    post_process=True,                          # bool: whether to apply face alignment post-processing (True)
    select_largest=True,                        # bool: if multiple faces are detected, return only the largest one (True)
    selection_method="center_weighted_size",    # str: heuristic for choosing among multiple detections—
                                                #   options include "largest" or "center_weighted_size" (default="center_weighted_size")
    keep_all=False,                             # bool: if True, return all detected faces; if False, return only one (default=False)
    device=device                               # torch.device or str: computation device, e.g. "cuda:0" or "cpu"
)


In this section, we prepare the test set so it can be efficiently used during inference. The aim is to create a `DataLoader` that iterates over face images, optionally applying preprocessing steps, and associates each image with the correct identity label.

The process includes the following components:

- **Image Transformations**  
  A basic image preprocessing pipeline is defined using `torchvision.transforms`. If MTCNN alignment is not enabled, each image is resized to 160×160 pixels (the input size required by the face recognition model) and then converted to a tensor.

- **Dataset Definition via ImageFolder**  
  The test images are loaded using `ImageFolder`, which expects the directory structure to be organised such that each identity has its own folder. The dataset automatically assigns a numerical label to each subfolder and loads all images accordingly.

- **Label Mapping**  
  A custom mapping `idx_to_class` is constructed to associate the internal numeric labels used by `ImageFolder` with the actual identity names provided in the test set metadata. This ensures label consistency throughout the evaluation process.

- **Final DataLoader**  
  The dataset is wrapped in a `DataLoader` for iteration. No parallel workers (`num_workers=0`) are used to maintain compatibility and simplicity. This `DataLoader` can now be used to either:
  - Apply face detection and alignment via MTCNN (if enabled), or
  - Pass pre-aligned images directly to the recognition model.

This structure ensures that the test set is handled in a clean, modular, and reproducible way.

In [ ]:
# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing_nn1 else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                  

The aligned dataset, which was stored in a PyTorch `DataLoader`, includes both the face images and their corresponding identity labels. Once loaded, the individual samples are unpacked and converted into two tensors: one containing all the aligned images, and the other containing the associated class labels.

To ensure compatibility with downstream processing steps, the image tensor is converted from PyTorch format to a NumPy array. This is especially useful when the data needs to be passed into frameworks such as **ART (Adversarial Robustness Toolbox)**, which often expect NumPy inputs for generating adversarial examples.

In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset/dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

## 2. Pre-Processing Defense Strategy

Based on the project guidelines that required the implementation of a **proactive defence strategy**, we adopted an approach based on **input pre-processing**. This method aligns with the principles of proactive security by attempting to **neutralise adversarial perturbations before they reach the model**, without relying on explicit adversarial example detection.

In particular, we implemented a **composite defence pipeline** inspired by the work of Xu et al. (2017), *“Feature Squeezing: Detecting Adversarial Examples in Deep Neural Networks”* ([arXiv:1704.01155](https://arxiv.org/pdf/1704.01155)). The authors show that certain transformations can effectively reduce input complexity and remove adversarial noise by discarding unnecessary input variance. These methods are also officially implemented in the [Adversarial Robustness Toolbox (ART)](https://github.com/Trusted-AI/adversarial-robustness-toolbox) under the `art.defences.preprocessor` module.

The pre-processing defence consists of the following sequential transformations:

- **JPEG Compression**: removes high-frequency noise by compressing the input image (default quality = 50). Adversarial perturbations often reside in fine-grained details that compression can discard.

- **Feature Squeezing**: reduces the bit-depth of input images (set to 4 bits), thereby lowering the input precision and limiting the attacker’s manipulation space.

- **Spatial Smoothing**: applies a median filter over a sliding window (size 11) to suppress localised noise while preserving semantic content.

These techniques are integrated using the helper function `preprocess_adversarial_inputs`, which handles the full pre-processing pipeline. It normalises the inputs, applies the transformations in sequence, and restores the resulting images to the standard input range expected by the model (`[-1, 1]`).

After processing, the resulting inputs are passed to the original model (`NN1`) without retraining or architectural changes. The function `build_dataloader_from_adversarial` is used to prepare the data in the correct format, ensuring that labels are mapped and inputs are correctly batched.

This setup enables a clear evaluation of the **restored classification accuracy** and the extent to which the defence **mitigates the effect of adversarial attacks**. Both performance metrics and visual examples are examined, allowing us to assess the robustness improvements achieved by this pre-processing shield.


In [ ]:
from art.defences.preprocessor import JpegCompression, FeatureSqueezing, SpatialSmoothing
import torch

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

### Baseline Model Evaluation for NN1 on Pre-Processed Clean Data

Once the composite pre-processing pipeline has been applied to the input data, we proceed to evaluate the effectiveness of the defence by running inference on the same model (NN1) using the processed samples.

The evaluation is conducted, as done before, using the helper function `evaluate_model`, which takes a `DataLoader` prepared via `build_dataloader_from_adversarial` to ensures that both the inputs and their corresponding labels are properly formatted for the model.

After this step, two performance metrics are then computed:

- **Classification Accuracy**: Measures the overall ability of the model to correctly classify inputs after the pre-processing defence has been applied.

- **Misclassification Analysis**: We log the most frequent errors to better understand which identities are still confused by the model, offering insight into the residual vulnerabilities that persist despite the defence.

In [ ]:
x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_aligned_nn1,
    jpeg_quality=25,
    bit_depth=5,
    window_size=7,
    labels=y_test_aligned_nn1
)

y_true = torch.load("y_true.pt")

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed,
    y_true=y_true,
    class_to_idx=class_to_idx,
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

correct, incorrect = print_basic_metrics(y_true, y_pred)
clean_acc_nn1 = accuracy_score(y_true, y_pred)
print(f"NN1 clean accuracy: {clean_acc_nn1*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred=y_pred,
    y_pred_adv=y_pred,
    title="Evaluation of NN1 on Preprocessed Samples (Spatial Smoothing)",
    id_test_images=idx_test_images,
    image_idx=0
)

### Attacks Setup

The `PyTorchClassifier` instance `classifier_nn1`, previously defined for evaluating model robustness, is reused here for the **systematic generation of adversarial examples**. By preserving the same classifier configuration—model architecture, loss function, input constraints, and class mapping—we ensure consistency across all attack experiments.

This setup enables us to **vary the parameters** of each adversarial attack (such as `ε`, `step size`, or `number of iterations`) without redefining the model or the training context.

In [ ]:
import torch.nn as nn
import torch.optim as optim

classifier_nn1 = PyTorchClassifier(
    model=nn1,                                                     # The PyTorch model to use
    clip_values=(-1, 1),                                           # The minimum and maximum values of the input
    loss=nn.CrossEntropyLoss(),                                    # The loss function
    optimizer=optim.Adam(nn1.parameters(), lr=0.01),               # The optimizer
    input_shape=(3, 160, 160),                                     # The shape of the input
    nb_classes=LABELS.size,                                        # The number of classes
    device_type='cuda' if torch.cuda.is_available() else 'cpu'
)

attack_folder = os.path.join(results_folder, 'attack_results_nn1')

### FGSM (Fast Gradient Sign Method) Adversarial Attack

We now reapply the **Fast Gradient Sign Method (FGSM)** adversarial attack, which was originally generated to assess the vulnerability of the model in its standard, undefended setting. This allows us to **re-use previously computed adversarial examples** and evaluate how well the defence pipeline—based on input pre-processing—can recover classification performance under attack.

By testing the model on these pre-generated adversarial inputs (created without any defence in place), we are able to **isolate and measure the benefit of the defence mechanism alone**, without the confounding effects of changing attack conditions.

In [ ]:
from art.attacks.evasion import FastGradientMethod
from art.defences.preprocessor import FeatureSqueezing, SpatialSmoothing, JpegCompression

fgsm_adv_folder = os.path.join(adversarial_folder, 'FGSM_nn1')
os.makedirs(fgsm_adv_folder, exist_ok=True)

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

We begin the evaluation of our defence strategy by testing it against adversarial examples generated using the FGSM attack in the `error generic setting`.

The adversarial samples used here were originally generated to assess model performance under attack, and are now re-evaluated after being processed through our pre-defined pre-processing pipeline. This pipeline includes **JPEG compression**, **feature squeezing**, and **spatial smoothing**, and is implemented via the `preprocess_adversarial_inputs` function.

After transformation, the inputs are passed to the original classifier (`NN1`), and standard evaluation metrics such as accuracy and prediction consistency are computed. This allows us to quantify the degree to which the input transformations can restore the classifier’s robustness.

In [ ]:
x_test_adv_fgsm_g = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_g.pt')
y_test_adv_fgsm_g = torch.load(fgsm_adv_folder + '/y_test_adv_fgsm_g.pt')

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_fgsm_g,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_fgsm_g
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

correct, incorrect = print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_test_adv_fgsm_g)
clean_acc_nn1 = accuracy_score(y_true, y_pred)
print(f"NN1 accuracy after defense against FGSM Error Generic: {clean_acc_nn1*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_test_adv_fgsm_g,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="FGSM (Error Generic) attack on NN1 with Pre-Processing Defense Techniques on NN1",
    image_idx=0,
)

We continue the robustness assessment by constructing a **Security Evaluation Curve (SEC)** for FGSM in the `error generic setting`. The goal is to systematically analyse how the model’s accuracy degrades as the strength of the attack increases.

To achieve this, a series of adversarial datasets are generated using the FGSM attack with increasing values of `ε`, which controls the magnitude of perturbation. Each adversarial set is then passed through the same defence pipeline described earlier: **JPEG compression**, **feature squeezing**, and **spatial smoothing**.

By evaluating the defended inputs at each attack level, we can trace how well the pre-processing strategy holds up against stronger adversarial perturbations. This produces a curve that visually depicts the model’s **residual accuracy** under defence, offering a practical measure of its robustness across a range of adversarial intensities.

The curve helps quantify the effectiveness of the defence in a fine-grained manner and provides insight into how sensitive the model remains to even mild perturbations despite the mitigation strategy.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Generic)
# ──────────────────────────────────────────────────────────────────────────────

# Valori di epsilon da testare
epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]

print("=== Security Evaluation Curve - FGSM (Error Generic) WITH DEFENSE ===\n")

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
accuracies = [0.98]

print("=== SEC with Defense: FGSM vs ε ===")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps)
    x_adv = fgsm.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )
    
    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)
    print(f"   Accuracy: {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
epsilons = [0] + epsilons
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (epsilons, accuracies, 'o', 'crimson', 'NN1')
    ],
    title="FGSM Error Generic - Security Evaluation Curve (Error Generic) \nwith Pre-Processing Defense Techniques on NN1",
    xlabel="Epsilon",
    ylabel="Accuracy"
)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

#### Error Specific

Continuing with the evaluation in the `error specific setting`, this stage focuses on targeted adversarial attacks using FGSM, where the objective is to mislead the model into predicting a fixed class — **'Dave_Mustaine'** — for all inputs, regardless of their true label.

A one-hot encoded vector representing the target class is generated and used to guide the attack. A reference image of the selected identity is displayed to offer a visual anchor for the intended classification outcome. The perturbation strength is set with `ε = 0.1`.

In [ ]:
fgsm_epsilon = 0.1

# Name of the target class
target_name = 'Dave_Mustaine'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

In [ ]:
x_test_adv_fgsm_s = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_s.pt')
y_test_adv_fgsm_s = torch.load(fgsm_adv_folder + '/y_test_adv_fgsm_s.pt')

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_fgsm_s,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_fgsm_s
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

acc = accuracy_score(y_true, y_pred)
sr  = (np.array(y_pred) == target_name).mean() * 100

print("=== FGSM Error Specific (Targeted) with Pre-Processing Defensive Techniques on NN1 ===")
print(f"Target: {target_name} | epsilon = {fgsm_epsilon}\n")
print(f"NN1 accuracy after defense against  {acc*100:.2f}%")
print(f"Targeted Success       : {sr:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true=y_true,
    y_pred=y_pred,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_fgsm_s
)

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_pred,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="FGSM (Error Specific) attack on NN1 with Pre-Processing Defense Techniques",
    image_idx=0,
)

As in the error generic scenario, `Security Evaluation Curves (SEC)` are used here to assess the model’s robustness against increasing levels of adversarial perturbation, controlled by the epsilon parameter, in the error specific case using the defence pipeline.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Specific)
# ──────────────────────────────────────────────────────────────────────────────

epsilons = [0.01, 0.02, 0.03, 0.05, 0.1]

print("=== Security Evaluation Curves - FGSM (Error Specific) WITH DEFENSE ===")

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
success_rates = [0.01]
accuracies = [0.98]

print("→ SEC with Defense: FGSM vs ε")
for eps in epsilons:
    print(f"\n→ Generating adversarial examples with ε = {eps:.3f}")
    fgsm = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=True)
    x_adv = fgsm.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true, y_pred = evaluate_model(nn1, loader, LABELS)
    
    sr = (np.array(y_pred) == target_name).mean()
    success_rates.append(sr)
    print(f"   Success Rate: {sr*100:.2f}%")

    acc = accuracy_score(y_true, y_pred)
    accuracies.append(acc)
    print(f"   Accuracy All:  {acc*100:.2f}%")

    _, _ = print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_adv)

# Plot finale e salvataggio
fig, ax = plt.subplots(figsize=(8, 5))

plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (epsilons, success_rates, 'o', 'crimson', 'Success Rate (Targeted)'),
        (epsilons, accuracies, 's', 'blue', 'Accuracy (All Classes)')
    ],
    title="FGSM - Security Evaluation Curve (Targeted vs All) with Defense Techniques on NN1 ",
    xlabel="Epsilon",
    ylabel="Metric"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

### BIM (Basic Iterative Method) Adversarial Attack 

Proceeding with the second adversarial attack, we now focus on the **Basic Iterative Method (BIM)**, which was previously used to assess the model’s vulnerability in the undefended scenario. As with FGSM, we reuse the **already generated adversarial examples** created under the original conditions—without any defence mechanism applied.

In [ ]:
from art.attacks.evasion import BasicIterativeMethod
from art.defences.preprocessor import FeatureSqueezing, SpatialSmoothing, JpegCompression

bim_adv_folder = os.path.join(adversarial_folder, 'BIM_nn1')
os.makedirs(bim_adv_folder, exist_ok=True)

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

Using the same evaluation structure adopted for FGSM, we apply the full pre-processing pipeline to adversarial examples generated by the **Basic Iterative Method (BIM)** in the `error generic setting`. After preprocessing, the classifier is re-evaluated to measure how effectively the defence recovers from the attack. Accuracy is computed along with a breakdown of correct and incorrect predictions.

This step helps assess whether the defence pipeline retains its effectiveness when facing stronger, iterative perturbations like those introduced by BIM.

In [ ]:
x_test_adv_bim_g = torch.load(bim_adv_folder + '/x_test_adv_bim_g.pt')
y_test_adv_bim_g = torch.load(bim_adv_folder + '/y_test_adv_bim_g.pt')

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_bim_g,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_bim_g
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

correct, incorrect = print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_test_adv_bim_g)
clean_acc_nn1 = accuracy_score(y_true, y_pred)
print(f"NN1 accuracy after defense against BIM: {clean_acc_nn1*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_test_adv_bim_g,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="BIM (Error Generic) attack on NN1 with Pre-Processing Defense Techniques",
    image_idx=0,
)

Following the FGSM analysis, this section performs a detailed robustness evaluation under the Basic Iterative Method (BIM) attack using `Security Evaluation Curves (SEC)`. The objective is to assess how the model responds when different parameters of BIM are varied, while the defence pipeline remains active.

Specifically, the SECs examine the effect of `three main BIM parameters` on classification accuracy: the overall `perturbation magnitude` (`ε`), the `step size` used at each iteration (`ε_step`), and the `total number of iterations` (`max_iter`).

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Generic) with Defense
# —————————————————————————————————————————————————————————————

eps_values       = [0.01, 0.03, 0.05, 0.1]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]

bim_eps          = 0.03
eps_step         = 0.01
max_iter         = 3

curve_colors = ['crimson', 'darkorange', 'seagreen']

print("=== Security Evaluation Curves - BIM (Error Generic) WITH DEFENSE ===")

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
accuracies_eps  = [0.98]

print("=== SEC with Defense: BIM vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples eps = {bim_eps:.3f}, eps_step = {eps_step:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=eps, eps_step=eps_step, max_iter=max_iter)
    x_adv = bim.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )
    
    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred_eval = evaluate_model(nn1, loader, LABELS)
    acc = accuracy_score(y_true_eval, y_pred_eval)
    accuracies_eps.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_eval, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_values, accuracies_eps, 'o', curve_colors[0], 'NN1 + Defense')],
    title=f"Accuracy vs ε with (ε_step={eps_step}, iter={max_iter})",
    xlabel="ε",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
accuracies_step  = [0.98]

print("\n=== SEC with Defense: BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {eps:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=eps, eps_step=step, max_iter=max_iter)
    x_adv = bim.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )
    
    y_true_eval, y_pred_eval = evaluate_model(nn1, loader, LABELS)
    acc = accuracy_score(y_true_eval, y_pred_eval)
    accuracies_step.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_eval, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_step_values, accuracies_step, 's', curve_colors[1], 'NN1 + Defense')],
    title=f"Accuracy vs ε_step with (ε={bim_eps}, iter={max_iter})",
    xlabel="ε step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
accuracies_iter   = [0.98]

print("\n=== SEC with Defense: BIM vs max_iter ===")
for n_iter in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {n_iter}, eps = {bim_eps:.3f}, eps_step = {eps_step:.3f}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=bim_eps, eps_step=eps_step, max_iter=n_iter)
    x_adv = bim.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )
    
    y_true_eval, y_pred_eval = evaluate_model(nn1, loader, LABELS)
    acc = accuracy_score(y_true_eval, y_pred_eval)
    accuracies_iter.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred_eval, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(max_iter_values, accuracies_iter, '^', curve_colors[2], 'NN1 + Defense')],
    title=f"Accuracy vs Max Iterations with (ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

#### Error Specific

Continuing the evaluation under `error specific conditions`, we focus on targeted misclassification towards the predefined class of: **Fernando_Torres**; and as always a corresponding one-hot encoded label is generated and replicated to guide subsequent adversarial attacks toward this specific identity.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

In [ ]:
x_test_adv_bim_s = torch.load(bim_adv_folder + '/x_test_adv_bim_s.pt')
y_test_adv_bim_s = torch.load(bim_adv_folder + '/y_test_adv_bim_s.pt')

bim_eps       = 0.03
bim_eps_step  = 0.01
bim_max_iter  = 3

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_bim_s,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_bim_s
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

acc = accuracy_score(y_true, y_pred)
sr  = (np.array(y_pred) == target_name).mean() * 100

print("=== BIM Error Specific (Targeted) with Defensive Techniques on NN1 ===")
print(f"Target: {target_name} | epsilon = {bim_eps} | epsilon_step = {bim_eps_step} | max_iter = {bim_max_iter}\n")
print(f"NN1 accuracy after defense against  {acc*100:.2f}%")
print(f"Targeted Success       : {sr:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true=y_true,
    y_pred=y_pred,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_bim_s
)

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_pred,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="BIM (Error Specific) attack on NN1 with Pre-Processing Defense Techniques",
    image_idx=0,
)

To complete the robustness analysis against BIM in the error specific setting, we construct a set of **Security Evaluation Curves (SEC)**. These curves illustrate how the **targeted success rate** varies when main attack parameters are changed, even in the presence of the full pre-processing defence pipeline.

Three experiments are conducted:

- **Varying ε**: Measures how increasing the maximum allowed perturbation impacts the defence’s ability to suppress targeted misclassification.

- **Varying ε_step**: Assesses how the step size of each BIM iteration influences the attack's success under defence.

- **Varying max_iter**: Evaluates the cumulative effect of more iterative refinement steps on the attack’s effectiveness, when defences are applied.

All adversarial examples are generated using the targeted variant of BIM and then passed through the composite defence pipeline (JPEG compression, feature squeezing, and spatial smoothing) before evaluation. The resulting targeted success rates reveal the **residual vulnerability of the model** after defence and help quantify the limits of its robustness.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Specific) with Defense
# —————————————————————————————————————————————————————————————

# Parameters
eps_values      = [0.01, 0.03, 0.05, 0.1]
eps_step_values = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values = [1, 3, 5, 10, 15]

bim_eps         = 0.03
eps_step        = 0.01
max_iter        = 3

print("=== Security Evaluation Curves - BIM (Error Specific) WITH DEFENSE ===")

curve_colors    = ['crimson', 'blue']

# —————————————————————————————————————————————————————————————
# 1) Accuracy & Targeted Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps         = [0.98]
sr_eps          = [0.01]

print("=== SEC with Defense (Error Specific): BIM vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples eps = {bim_eps:.3f}, eps_step = {eps_step:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=eps, eps_step=eps_step, max_iter=max_iter, targeted=True)
    x_adv = bim.generate(x_test_aligned_nn1, one_hot_targeted_label)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )
    
    y_true, y_pred = evaluate_model(nn1, loader, LABELS)
    acc = accuracy_score(y_true, y_pred)
    sr = (np.array(y_pred) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Success Rate: {sr*100:.2f}%")
    print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Epsilon vs Accuracy / Targeted Accuracy\nPreProcess+NN1 (ε_step={eps_step}, iter={max_iter})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy & Targeted Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step        = [0.98]
sr_step         = [0.01]

print("\n=== SEC with Defense (Error Specific): BIM vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples eps_step = {step:.3f}, eps = {bim_eps:.3f}, max_iter = {max_iter}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=bim_eps, eps_step=step, max_iter=max_iter, targeted=True)
    x_adv = bim.generate(x_test_aligned_nn1, one_hot_targeted_label)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )
    
    y_true, y_pred = evaluate_model(nn1, loader, LABELS)
    
    acc = accuracy_score(y_true, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Success Rate: {sr*100:.2f}%")
    print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Epsilon Step vs Accuracy / Targeted Accuracy\nPreProcess+NN1 (ε={eps}, iter={max_iter})",
    xlabel="Epsilon Step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy & Targeted Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter    = [0.98]
sr_iter     = [0.01]

print("\n=== SEC with Defense (Error Specific): BIM vs max_iter ===")
for n_iter in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {n_iter}, eps = {bim_eps:.3f}, eps_step = {eps_step:.3f}")
    bim = BasicIterativeMethod(estimator=classifier_nn1, eps=bim_eps, eps_step=eps_step, max_iter=n_iter, targeted=True)
    x_adv = bim.generate(x_test_aligned_nn1, one_hot_targeted_label)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )
    
    y_true, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")
    print_basic_metrics(y_true, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Max Iterations vs Accuracy / Targeted Accuracy\nPreProcess+NN1 (ε={eps}, ε_step={eps_step})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

### PGD (Projected Gradient Descent) Adversarial Attack

Following the evaluations with FGSM and BIM, this section prepares the environment for testing the model against the `Projected Gradient Descent (PGD) attack`.

In [ ]:
from art.attacks.evasion import ProjectedGradientDescent
from art.defences.preprocessor import FeatureSqueezing, SpatialSmoothing, JpegCompression

pgd_adv_folder = os.path.join(adversarial_folder, 'PGD_nn1')
os.makedirs(pgd_adv_folder, exist_ok=True)

y_true = torch.load('y_true.pt')

#### Error Generic

Continuing the robustness analysis, this section evaluates the model under a `Projected Gradient Descent (PGD) attack` in the `error generic setting`. Adversarial examples are processed through the established defence pipeline before classification. The resulting accuracy and error metrics are used to assess the model’s resilience to PGD, a more powerful iterative attack with random initialisation, under the same evaluation framework used for FGSM and BIM.

In [ ]:
x_test_adv_pgd_g = torch.load(pgd_adv_folder + '/x_test_adv_pgd_g.pt')
y_test_adv_pgd_g = torch.load(pgd_adv_folder + '/y_test_adv_pgd_g.pt')

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
num_random_init = 5

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_pgd_g,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    model=classifier_nn1,
    label_map=LABELS
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true_eval, y_pred= evaluate_model(nn1, loader, LABELS)

acc_defended = accuracy_score(y_true_eval, y_pred)

print("=== PGD Evaluation (Error Generic) with Defense ===")
print(f"ε = {pgd_eps}, ε_step = {pgd_eps_step}, iter = {pgd_max_iter}, init = {num_random_init}")
print(f"NN1 accuracy after defense against  {acc_defended*100:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true_eval,
    y_pred,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_pgd_g
)

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_smoothed,
    y_true         = y_true_eval,
    y_pred         = y_pred,
    y_pred_adv     = y_test_adv_pgd_g,
    title          = f"PGD Attack (Error Generic) with Pre-Processing Defense - ε={pgd_eps}",
    id_test_images = idx_test_images,
    image_idx      = 0
)

As done with FGSM and BIM, this section evaluates the model’s `robustness under the PGD attack` using `Security Evaluation Curves (SEC)`. The same defence pipeline is applied, and model accuracy is measured while varying PGD parameters: `perturbation size` (`ε`), `step size` (`ε_step`), `number of iterations` (`max_iter`), and `number of random initialisations` (`num_random_init`).

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - PGD (Error Generic) with Defense
# —————————————————————————————————————————————————————————————

eps_values       = [0.01, 0.03, 0.05, 0.10]
eps_step_values  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_values  = [1, 3, 5, 10, 15]
num_init_values  = [1, 3, 5]

pgd_eps          = 0.03
pgd_eps_step     = 0.01
pgd_max_iter     = 3
num_random_init  = 5

curve_colors     = ['crimson', 'darkorange', 'seagreen', 'royalblue']

print("=== Security Evaluation Curves - PGD (Error Generic) WITH DEFENSE ===")

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps = [0.98]

print("=== SEC with Defense: PGD vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples ε = {eps:.3f}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_eps.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_values, acc_eps, 'o', curve_colors[0], 'NN1 + Defense')],
    title=f"Accuracy vs ε (ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)


# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step = [0.98]

print("\n=== SEC with Defense: PGD vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples ε_step = {step:.3f}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_step.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(eps_step_values, acc_step, 's', curve_colors[1], 'NN1 + Defense')],
    title=f"Accuracy vs ε_step (ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="ε step",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter = [0.98]

print("\n=== SEC with Defense: PGD vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=it,
        num_random_init=num_random_init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)

    # Difesa
    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_iter.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(max_iter_values, acc_iter, '^', curve_colors[2], 'NN1 + Defense')],
    title=f"Accuracy vs Max iterations (ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max iterations",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
acc_init = [0.98]

print("\n=== SEC with Defense: PGD vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples num_random_init = {init}")
    pgd = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=init,
        targeted=False
    )
    x_adv = pgd.generate(x=x_test_aligned_nn1)

    x_test_smoothed = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    acc_init.append(acc)
    print(f"   Accuracy after defense : {acc*100:.2f}%")
    _, _ = print_basic_metrics(y_true_eval, y_pred, x_test_aligned_nn1, x_adv)

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[(num_init_values, acc_init, 'd', curve_colors[3], 'NN1 + Defense')],
    title=f"Accuracy vs Random Initializations (ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

#### Error Specific

We now proceed to assess the PGD attack in the `error specific scenario`, targeting the class 'Fernando_Torres'.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

Moving on we test the effectiveness of the pre-processing defence pipeline against adversarial examples generated using **Projected Gradient Descent (PGD)** in the targeted setting.

The perturbed inputs, crafted to mislead the model into predicting a fixed identity, are processed through the same three-stage transformation (JPEG compression, bit-depth reduction, and spatial smoothing). The resulting predictions are then compared to the ground-truth labels to assess the defence's ability to mitigate targeted attacks.

In [ ]:
x_test_adv_pgd_s = torch.load(pgd_adv_folder + '/x_test_adv_pgd_s.pt')
y_test_adv_pgd_s = torch.load(pgd_adv_folder + '/y_test_adv_pgd_s.pt')

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 3
num_random_init = 5

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_pgd_s,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    model=classifier_nn1,
    label_map=LABELS
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true_eval, y_pred= evaluate_model(nn1, loader, LABELS)

acc_defended = accuracy_score(y_true_eval, y_pred)

print("=== PGD Specific (Targeted) Evaluation with Defense ===")
print(f"Target class       : {target_name}")
print(f"ε = {pgd_eps}, ε_step = {pgd_eps_step}, iter = {pgd_max_iter}, init = {num_random_init}")
print(f"NN1 accuracy after defense against  {acc_defended*100:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true_eval,
    y_pred,
    x_orig = x_test_aligned_nn1,
    x_adv  = x_test_adv_pgd_s
)

plot_predicted_images(
    x_test         = x_test_aligned_nn1,
    x_adv          = x_test_smoothed,
    y_true         = y_true_eval,
    y_pred         = y_pred,
    y_pred_adv     = y_test_adv_pgd_s,
    title          = f"PGD Specific Attack with Pre-Processing Defense → Target: {target_name}",
    id_test_images = idx_test_images,
    image_idx      = 0
)

Concluding the analysis, we assess how the model behaves under the **Projected Gradient Descent (PGD)** attack in the targeted setting when the **pre-processing defence pipeline** is applied.

Four Security Evaluation Curves (SEC) are generated, each exploring the effect of a different PGD parameter on both overall accuracy and targeted success rate.

In [ ]:
# —————————————————————————————————————————————————————————————
# PGD Targeted - Security Evaluation Curves (Error Specific) WITH DEFENSE
# —————————————————————————————————————————————————————————————

# Parametri
eps_values      = [0.005, 0.01, 0.03, 0.05, 0.10]
eps_step_values = [0.005, 0.01, 0.05]
max_iter_values = [3, 5, 10, 15]
num_init_values = [1, 3, 5]

pgd_eps         = 0.03
pgd_eps_step    = 0.01
pgd_max_iter    = 10
num_random_init = 5

curve_colors     = ['crimson', 'darkorange', 'seagreen', 'royalblue']

print("=== Security Evaluation Curves - PGD (Error Specific) WITH DEFENSE ===")

# —————————————————————————————————————————————————————————————
# 1) Accuracy & Targeted Accuracy vs ε
# —————————————————————————————————————————————————————————————
acc_eps, sr_eps = [0.98], [0.01]

print("=== SEC with Defense: PGD Targeted vs ε ===")
for eps in eps_values:
    print(f"\n→ Generating adversarial examples ε = {eps:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)

    x_def = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)
    acc = accuracy_score(y_true_eval, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_eps.append(acc)
    sr_eps.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
eps_values = [0] + eps_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_values, acc_eps, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_values, sr_eps,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon\nPreProcess+NN1 (ε_step={pgd_eps_step}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsilon",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy & Targeted Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
acc_step, sr_step = [0.98], [0.01]

print("\n=== SEC with Defense: PGD Targeted vs ε_step ===")
for step in eps_step_values:
    print(f"\n→ Generating adversarial examples ε_step = {step:.3f}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=step,
        max_iter=pgd_max_iter,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)

    x_def = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)
    
    acc = accuracy_score(y_true_eval, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_step.append(acc)
    sr_step.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
eps_step_values = [0] + eps_step_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (eps_step_values, acc_step, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (eps_step_values, sr_step,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Epsilon Step\nPreProcess+NN1 (ε={pgd_eps}, iter={pgd_max_iter}, init={num_random_init})",
    xlabel="Epsion Step",
    ylabel="Accuracy"
)


# —————————————————————————————————————————————————————————————
# 3) Accuracy & Targeted Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
acc_iter, sr_iter = [0.98], [0.01]

print("\n=== SEC with Defense: PGD Targeted vs max_iter ===")
for it in max_iter_values:
    print(f"\n→ Generating adversarial examples max_iter = {it}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=it,
        num_random_init=num_random_init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)

    x_def = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_iter.append(acc)
    sr_iter.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
max_iter_values = [0] + max_iter_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (max_iter_values, acc_iter, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (max_iter_values, sr_iter,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Max Iterations\nPreProcess+NN1 (ε={pgd_eps}, ε_step={pgd_eps_step}, init={num_random_init})",
    xlabel="Max Iterations",
    ylabel="Accuracy"
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy & Targeted Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
acc_init, sr_init = [0.98], [0.01]

print("\n=== SEC with Defense: PGD Targeted vs num_random_init ===")
for init in num_init_values:
    print(f"\n→ Generating adversarial examples num_random_init = {init}")
    attack = ProjectedGradientDescent(
        estimator=classifier_nn1,
        eps=pgd_eps,
        eps_step=pgd_eps_step,
        max_iter=pgd_max_iter,
        num_random_init=init,
        targeted=True
    )
    x_adv = attack.generate(x=x_test_aligned_nn1, y=one_hot_targeted_label)

    x_def = preprocess_adversarial_inputs(
        x_adv=x_adv,
        jpeg_quality=25
        bit_depth=5,
        window_size=7,
        model=classifier_nn1,
        label_map=LABELS
    )

    loader = build_dataloader_from_adversarial(
        x_adv=x_test_smoothed, 
        y_true=y_true, 
        class_to_idx=class_to_idx, 
        idx_to_class=dataset.idx_to_class
    )

    y_true_eval, y_pred = evaluate_model(nn1, loader, LABELS)

    acc = accuracy_score(y_true_eval, y_pred)
    sr  = (np.array(y_pred) == target_name).mean()
    acc_init.append(acc)
    sr_init.append(sr)
    print(f"   Accuracy: {acc*100:.2f}% | Targeted Accuracy: {sr*100:.2f}%")

fig, ax = plt.subplots(figsize=(8, 5))
num_init_values = [0] + num_init_values
plot_multiple_sec_curves(
    ax=ax,
    curves=[
        (num_init_values, acc_init, 'o', curve_colors[0], 'Accuracy (All Classes)'),
        (num_init_values, sr_init,  's', curve_colors[1], 'Success Rate (Targeted)'),
    ],
    title=f"Accuracy / Targeted Accuracy vs Random Initializations\nPreProcess+NN1 (ε={pgd_eps}, ε_step={pgd_eps_step}, iter={pgd_max_iter})",
    xlabel="Random Initializations",
    ylabel="Accuracy"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### Carlini Wagner (CW) $L_{\infty}$ - Adversarial Attack

To conclude our defence evaluation, we examine the performance of the pre-processing pipeline against adversarial examples generated using the **`Carlini & Wagner L∞ attack`**. This attack provides a final benchmark to assess the robustness enhancements introduced by the defence.

In [ ]:
cw_adv_folder = os.path.join(adversarial_folder, 'CW_L_inf_nn1')
os.makedirs(cw_adv_folder, exist_ok=True)

# Load the embeddings for the test set
y_true = torch.load('y_true.pt')

#### Error Generic

The model is now tested against the Carlini & Wagner attack in the `error generic setting`.

In [ ]:
x_test_adv_cw_g = torch.load(cw_adv_folder + '/x_test_adv_cw_g.pt')
y_test_adv_cw_g = torch.load(cw_adv_folder + '/y_test_adv_cw_g.pt')

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_cw_g,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_cw_g
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true, y_pred = evaluate_model(nn1, loader, LABELS)

correct, incorrect = print_basic_metrics(y_true, y_pred, x_test_aligned_nn1,x_test_adv_cw_g)
clean_acc_nn1 = accuracy_score(y_true, y_pred)
print(f"NN1 clean accuracy: {clean_acc_nn1*100:.2f}%")

mismatches = [(t, p) for t, p in zip(y_true, y_pred) if t != p]
print("\nNot Correctly classified:")
for (t, p), count in Counter(mismatches).most_common(10):
    print(f"❌ {t} → {p}  ({count} volte)")
    
plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_pred,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="Carlini Wagner (Error Generic) attack on NN1 with Pre-Processing Defense Techniques",
    image_idx=0,
)

#### Error Specific

And then we perform the final evaluation in the `error specific setting` using the Carlini & Wagner.

In [ ]:
# Name of the target class
target_name = 'Fernando_Torres'

one_hot_targeted_label = generate_one_hot_target_and_plot_image(target_name, LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

The model is now tested and evaluated with the previously generated attack on nn1 loaded 

In [ ]:
x_test_adv_cw_s = torch.load(cw_adv_folder + '/x_test_adv_cw_s.pt')
y_test_adv_cw_s = torch.load(cw_adv_folder + '/y_test_adv_cw_s.pt')

cw_target_conf     = 0.5
cw_max_iter        = 7

x_test_smoothed = preprocess_adversarial_inputs(
    x_adv=x_test_adv_cw_s,
    jpeg_quality=25
    bit_depth=5,
    window_size=7,
    labels=y_test_adv_cw_s
)

loader = build_dataloader_from_adversarial(
    x_adv=x_test_smoothed, 
    y_true=y_true, 
    class_to_idx=class_to_idx, 
    idx_to_class=dataset.idx_to_class
)

y_true_eval, y_pred= evaluate_model(nn1, loader, LABELS)

acc = accuracy_score(y_true, y_test_adv_cw_s)
sr  = (np.array(y_test_adv_cw_s) == target_name).mean() * 100

print("=== CW L∞ Error Specific (Targeted) ===")
print(f"Target: {target_name} | Confidence: {cw_target_conf} | Iter: {cw_max_iter}\n")
print(f"Accuracy           : {acc*100:.2f}%")
print(f"Targeted Success   : {sr:.2f}%\n")

correct, incorrect = print_basic_metrics(
    y_true=y_true,
    y_pred=y_test_adv_cw_s,
    x_orig=x_test_aligned_nn1,
    x_adv=x_test_adv_cw_s
)

plot_predicted_images(
    x_test=x_test_aligned_nn1,
    x_adv=x_test_smoothed,
    y_true=y_true,
    y_pred_adv=y_pred,
    y_pred=y_pred,
    id_test_images=idx_test_images,
    title="Carlini Wagner (Error Specific) attack on NN1 with Pre-Processing Defense Techniques",
    image_idx=0,
)